# GAT with SAGPool Session Readout

This notebook follows the SR-GNN / TAGNN session recommendation pipeline, but replaces the GGNN encoder with a Graph Attention Network and the readout with a SAGPool-style hierarchical pooling block.

After a clean run, `RESULTS_DIR` contains:

- `gat_sagpool_results.csv`, `gat_sagpool_history.csv`, `gat_sagpool_checkpoint_summary.csv`, `gat_sagpool_model_summary_rows.csv`
- `gat_sagpool_metrics.png`
- `checkpoints/gat_sagpool_<dataset>.pt` (best-by-`MRR@20`, plus final test metrics in the same file)
- updated `model_summary.csv` and `gat_three_way_comparison.csv` when prior baselines are present

## Architecture (target end state)

A session prefix `[v1, v2, ..., vt]` is turned into a graph. Nodes are unique items from the prefix. Directed edges follow observed clicks, for example `v1 -> v2`; reverse edges are also built so the model can read both incoming and outgoing transition context. Repeated transitions are normalized and passed as edge weights.

The encoder is a bidirectional GAT shared with the other two notebooks (same `BidirectionalGATEncoder` interface). The readout is replaced by SAGPool-style hierarchical pooling over the session graph:

```text
session prefix
     |
directed session graph
     |
item embedding, dim=100
     |
     +--> forward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
     |
     +--> backward GATConv: 4 heads x 25 dims, dropout=0.1, edge weights
                 |
concat forward/backward states, dim=200
                 |
linear direction fusion, 200 -> 100
                 |
contextual node embeddings
                 |
SAGPool readout : score nodes, keep top-k, aggregate
                 |
session representation
                 |
scores for all items
```

GAT details kept identical to the other two notebooks (so encoder differences are isolated only by the readout):

- Layer type: `torch_geometric.nn.GATConv`.
- Hidden size: `100`.
- Number of GAT layers per direction: `1`.
- Attention heads: `4`.
- Per-head output size: `25`, concatenated back to `100`.
- Dropout inside GAT and after activation: `0.1`.
- Activation: `ELU`.
- Residual connection: enabled inside each directional stack.
- Self-loops: enabled by `GATConv`.
- Edge features: normalized transition weights passed as one-dimensional edge attributes.
- Direction fusion: concatenate forward and backward node states, then apply a linear layer `200 -> 100`.

## Prediction (target end state)

The final session vector is multiplied by the item embedding matrix. This gives one score for every candidate item. Items are ranked by score, and the top 20 are used for `Precision@20` and `MRR@20`.

This notebook prepares raw Yoochoose and Diginetica inputs, runs the SR-GNN / TAGNN preprocessing pipeline, and is wired so that adding the SAGPool model requires only:

1. defining `GATSAGPool` class
2. providing a factory `lambda num_items, config: GATSAGPool(...)` to `run_dataset_experiment`

Expected Kaggle input directories:
- `/kaggle/input/datasets/chadgostopp/recsys-challenge-2015` containing `yoochoose-clicks.dat`
- `/kaggle/input/datasets/profalbusdumbledore/diginetica-dataset` containing `train-item-views.csv`

`Yoochoose 1/4` is skipped because it is too large for the target memory budget.

## Environment and Dependencies

Kaggle base image already provides `torch`, `pandas`, `numpy`, `matplotlib`, and `kagglehub`. The PyG stack (`torch_geometric`, used by GATConv / SAGPooling / GraphConv / `global_*_pool`) is not preinstalled, so it is installed here.

The install cell is idempotent (skips if `torch_geometric` is already importable) and quiet. From PyG `>=2.3` the base `pip install torch_geometric` is enough for everything this notebook uses; `pyg_lib`, `torch_scatter`, `torch_sparse` are not required.

Kaggle settings required for a clean first run:

- **Accelerator: GPU** (T4 is sufficient).
- **Internet: On** (otherwise the pip install fails).
- **Environment: latest** (Settings -> Environment -> "Always use latest environment") - avoids torch / torch_geometric version mismatches that occasionally appear with pinned-old environments.

After the install, the cell prints the resolved `torch` and `torch_geometric` versions plus `cuda_available`, so any environment problem is visible at the very top of the run instead of mid-training.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys


def _ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package = pip_name or import_name
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package],
            check=True,
        )
        importlib.invalidate_caches()


_ensure_package("torch_geometric")

import torch  # noqa: E402
import torch_geometric  # noqa: E402

print(f"torch={torch.__version__}")
print(f"torch_geometric={torch_geometric.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")

## Prepare Datasets


In [ ]:
import math
import os
import random
import time
from collections import Counter
from datetime import date
from pathlib import Path

MPLCONFIGDIR = Path("/tmp/matplotlib")
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import (
    GATConv,
    GraphConv,
    SAGPooling,
    global_max_pool,
    global_mean_pool,
)
from torch_geometric.utils import softmax as pyg_softmax

In [ ]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

YOOCHOOSE_INPUT_DIR = KAGGLE_INPUT_DIR / "datasets/chadgostopp/recsys-challenge-2015"
DIGINETICA_INPUT_DIR = (
    KAGGLE_INPUT_DIR / "datasets/profalbusdumbledore/diginetica-dataset"
)
YOOCHOOSE_SOURCE = YOOCHOOSE_INPUT_DIR / "yoochoose-clicks.dat"
DIGINETICA_SOURCE = DIGINETICA_INPUT_DIR / "train-item-views.csv"

RESULTS_DIR = OUTPUT_DIR / "results"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [RESULTS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"output_dir={OUTPUT_DIR}")
print(f"Using Yoochoose source: {YOOCHOOSE_SOURCE}")
print(f"Using Diginetica source: {DIGINETICA_SOURCE}")

## Preprocess Sessions

The preprocessing flow builds ordered sessions, removes short sessions and rare items, splits chronologically, remaps item ids from training data, and expands sessions into prefix-label examples.

In [ ]:
def load_yoochoose_sessions(path):
    df = pd.read_csv(
        path,
        header=None,
        usecols=[0, 1, 2],
        names=["session_id", "timestamp", "item_id"],
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(None)

    session_items = {}
    session_dates = {}
    current_session_id = None
    current_timestamp = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_timestamp is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_timestamp

        current_session_id = session_id
        current_timestamp = row.timestamp

        if session_id in session_items:
            session_items[session_id].append(row.item_id)
        else:
            session_items[session_id] = [row.item_id]

    if current_session_id is not None:
        session_dates[current_session_id] = current_timestamp

    return [
        (session_id, session_dates[session_id], items)
        for session_id, items in session_items.items()
    ]


def load_diginetica_sessions(path):
    df = pd.read_csv(
        path,
        sep=";",
        usecols=["sessionId", "itemId", "timeframe", "eventdate"],
    )
    df = df.rename(columns={"sessionId": "session_id", "itemId": "item_id"})
    df["eventdate"] = pd.to_datetime(df["eventdate"], format="%Y-%m-%d")

    session_clicks = {}
    session_dates = {}
    current_session_id = None
    current_date = None

    for row in df.itertuples(index=False):
        session_id = int(row.session_id)
        if current_date is not None and current_session_id != session_id:
            session_dates[current_session_id] = current_date

        current_session_id = session_id
        current_date = row.eventdate

        click = (row.item_id, int(row.timeframe))
        if session_id in session_clicks:
            session_clicks[session_id].append(click)
        else:
            session_clicks[session_id] = [click]

    if current_session_id is not None:
        session_dates[current_session_id] = current_date

    sessions = []
    for session_id, clicks in session_clicks.items():
        ordered_clicks = sorted(clicks, key=lambda click: click[1])
        items = [item for item, _ in ordered_clicks]
        sessions.append((session_id, session_dates[session_id], items))
    return sessions

In [ ]:
def drop_short_sessions(sessions):
    return [session for session in sessions if len(session[2]) >= 2]


def drop_rare_items(sessions, min_freq=5):
    counts = Counter()
    for _, _, items in sessions:
        counts.update(items)

    result = []
    for session_id, date, items in sessions:
        kept = [i for i in items if counts[i] >= min_freq]
        if len(kept) >= 2:
            result.append((session_id, date, kept))
    return result


def sort_by_date(sessions):
    return sorted(sessions, key=lambda session: session[1])


def split_by_date(sessions, test_days):
    max_date = max(date for _, date, _ in sessions)
    split_date = max_date - pd.Timedelta(days=test_days)
    train = [s for s in sessions if s[1] < split_date]
    test = [s for s in sessions if s[1] > split_date]
    return train, test


def renumber_training_items(train_sessions):
    item_to_index = {}
    next_item_index = 1
    remapped_sessions = []

    for session_id, date, items in train_sessions:
        remapped_items = []
        for item in items:
            if item not in item_to_index:
                item_to_index[item] = next_item_index
                next_item_index += 1
            remapped_items.append(item_to_index[item])
        remapped_sessions.append((session_id, date, remapped_items))

    return remapped_sessions, item_to_index


def remap_test_sessions(test_sessions, item_to_index):
    remapped_sessions = []
    for session_id, date, items in test_sessions:
        remapped_items = [
            item_to_index[item] for item in items if item in item_to_index
        ]
        if len(remapped_items) >= 2:
            remapped_sessions.append((session_id, date, remapped_items))
    return remapped_sessions


def expand_sessions(sessions):
    examples = []
    for session_id, date, items in sessions:
        for reverse_offset in range(1, len(items)):
            examples.append(
                (session_id, date, items[:-reverse_offset], items[-reverse_offset])
            )
    return examples


def keep_recent_fraction(examples, denominator):
    if denominator is None:
        return examples
    keep = len(examples) // denominator
    return examples[-keep:] if keep else examples


def prefix_label_rows(examples):
    return [(list(prefix), int(label)) for _, _, prefix, label in examples]


def vocabulary_size_from_rows(*row_groups):
    max_item_id = 0
    for rows in row_groups:
        for prefix, label in rows:
            max_item_id = max(max_item_id, int(label), max(prefix))
    return max_item_id + 1


def preprocess(sessions, test_days, train_fraction_denominator):
    sessions = drop_short_sessions(sessions)
    sessions = drop_rare_items(sessions)
    sessions = sort_by_date(sessions)
    train_sessions, test_sessions = split_by_date(sessions, test_days)

    train_sessions, item_to_index = renumber_training_items(train_sessions)
    test_sessions = remap_test_sessions(test_sessions, item_to_index)

    train_examples = expand_sessions(train_sessions)
    test_examples = expand_sessions(test_sessions)
    train_examples = keep_recent_fraction(train_examples, train_fraction_denominator)
    return train_examples, test_examples

In [ ]:
yoochoose_sessions = load_yoochoose_sessions(YOOCHOOSE_SOURCE)
diginetica_sessions = load_diginetica_sessions(DIGINETICA_SOURCE)

yoochoose_1_64_train, yoochoose_1_64_test = preprocess(
    yoochoose_sessions, test_days=1, train_fraction_denominator=64
)
diginetica_train, diginetica_test = preprocess(
    diginetica_sessions, test_days=7, train_fraction_denominator=None
)

YOOCHOOSE_1_64_TRAIN_ROWS = prefix_label_rows(yoochoose_1_64_train)
YOOCHOOSE_1_64_TEST_ROWS = prefix_label_rows(yoochoose_1_64_test)
DIGINETICA_TRAIN_ROWS = prefix_label_rows(diginetica_train)
DIGINETICA_TEST_ROWS = prefix_label_rows(diginetica_test)

print(f"Yoochoose 1/64 train examples: {len(YOOCHOOSE_1_64_TRAIN_ROWS):,}")
print(f"Yoochoose 1/64 test examples: {len(YOOCHOOSE_1_64_TEST_ROWS):,}")
print(f"Diginetica train examples: {len(DIGINETICA_TRAIN_ROWS):,}")
print(f"Diginetica test examples: {len(DIGINETICA_TEST_ROWS):,}")

In [ ]:
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

if torch.cuda.is_available():
    print(f"cuda_device={torch.cuda.get_device_name(0)}")
    print(
        f"cuda_memory_gb={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}"
    )
else:
    print("cuda_device=None")

print("Available Kaggle input files:")
for dirname, _, filenames in os.walk(KAGGLE_INPUT_DIR):
    for filename in filenames:
        print(Path(dirname) / filename)

## Data loading and graph construction helpers

The session-graph contract is intentionally identical to `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`. This guarantees that the only thing changing across the three GAT branches is the readout (and, by extension, the part of the model after the encoder), so any difference in `Precision@20` / `MRR@20` is attributable to the readout, not to graph construction.

Each session prefix becomes a `torch_geometric.data.Data` object with:

- `x`: original training-vocabulary item IDs for the unique nodes in the prefix; this is the input to the shared item embedding table.
- `edge_index` / `edge_weight`: forward (click order) edges with normalized transition weights (default channel used by PyG mini-batch utilities).
- `forward_edge_index` / `forward_edge_weight`: forward transition edges with normalized weights.
- `backward_edge_index` / `backward_edge_weight`: reverse transition edges with normalized weights.
- `sequence`: per-position local node indices, so contextual node embeddings can be re-aligned with the original click order at readout time.
- `sequence_length`: length of the original prefix (used during batched scatter / pooling operations).
- `last_click`: local node index of the last clicked item in the prefix.
- `y`: integer label (the true next item ID).
- `num_nodes`: number of unique items in the prefix.

The dataset is lazy: graphs are built on demand inside `__getitem__`, so we do not materialize all graphs in memory before training starts.

In [ ]:
def build_session_graph(prefix, label):
    """Turn a `(prefix, label)` example into a PyG `Data` graph.

    Nodes are unique items in the prefix. Forward edges follow click order;
    backward edges expose reverse transition context, mirroring the
    SR-GNN / TAGNN incoming/outgoing adjacency channels.
    """
    unique_items = list(dict.fromkeys(prefix))
    item_to_node = {item: index for index, item in enumerate(unique_items)}
    click_sequence = [item_to_node[item] for item in prefix]

    edge_counts = Counter(zip(click_sequence[:-1], click_sequence[1:]))
    out_degree = Counter()
    in_degree = Counter()
    for (source, target), count in edge_counts.items():
        out_degree[source] += count
        in_degree[target] += count

    forward_sources, forward_targets, forward_weights = [], [], []
    backward_sources, backward_targets, backward_weights = [], [], []
    for (source, target), count in edge_counts.items():
        forward_sources.append(source)
        forward_targets.append(target)
        forward_weights.append(count / out_degree[source])

        backward_sources.append(target)
        backward_targets.append(source)
        backward_weights.append(count / in_degree[target])

    if forward_sources:
        forward_edge_index = torch.tensor(
            [forward_sources, forward_targets], dtype=torch.long
        )
        forward_edge_weight = torch.tensor(forward_weights, dtype=torch.float)
        backward_edge_index = torch.tensor(
            [backward_sources, backward_targets], dtype=torch.long
        )
        backward_edge_weight = torch.tensor(backward_weights, dtype=torch.float)
    else:
        forward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        forward_edge_weight = torch.zeros(0, dtype=torch.float)
        backward_edge_index = torch.zeros((2, 0), dtype=torch.long)
        backward_edge_weight = torch.zeros(0, dtype=torch.float)

    return Data(
        x=torch.tensor(unique_items, dtype=torch.long),
        edge_index=forward_edge_index,
        edge_weight=forward_edge_weight,
        forward_edge_index=forward_edge_index,
        forward_edge_weight=forward_edge_weight,
        backward_edge_index=backward_edge_index,
        backward_edge_weight=backward_edge_weight,
        sequence=torch.tensor(click_sequence, dtype=torch.long),
        sequence_length=torch.tensor(len(click_sequence), dtype=torch.long),
        last_click=torch.tensor(click_sequence[-1], dtype=torch.long),
        y=torch.tensor(label, dtype=torch.long),
        num_nodes=len(unique_items),
    )


class SessionGraphDataset(Dataset):
    """Lazy dataset of session-prefix graphs.

    Graphs are built on demand so training does not materialize all PyG
    objects in memory before training starts.
    """

    def __init__(self, prefix_label_rows):
        self.prefix_label_rows = prefix_label_rows

    def __len__(self):
        return len(self.prefix_label_rows)

    def __getitem__(self, index):
        prefix, label = self.prefix_label_rows[index]
        return build_session_graph(prefix, label)

## Vocabulary size

The item embedding table dimension is derived from the maximum remapped item id seen in any train/test row, plus one for padding (id `0`). This must match the convention used by `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`, otherwise the resulting checkpoints and metrics are not directly comparable.

In [ ]:
yoochoose_1_64_num_items = vocabulary_size_from_rows(
    YOOCHOOSE_1_64_TRAIN_ROWS, YOOCHOOSE_1_64_TEST_ROWS
)
diginetica_num_items = vocabulary_size_from_rows(
    DIGINETICA_TRAIN_ROWS, DIGINETICA_TEST_ROWS
)

print(f"Yoochoose 1/64 num_items: {yoochoose_1_64_num_items:,}")
print(f"Diginetica      num_items: {diginetica_num_items:,}")

## Training and Evaluation Protocol

The training/evaluation contract here intentionally mirrors `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb` so the three branches share the same loop, the same metric definitions, and the same artifact format under `results/` and `results/checkpoints/`.

Setup carried over from SR-GNN / TAGNN (matching the SR-GNN paper protocol):

- Hidden size `100`.
- Random `10%` validation split from the training set.
- Adam, learning rate `0.001`.
- Learning-rate decay by `0.1` every `3` epochs.
- Batch size `100`.
- L2 penalty `1e-5`.
- Early stopping on validation `MRR@20` with patience `5`.
- Padding item id `0` is masked out before ranking metrics.

Helpers exposed (same names as in the other two notebooks for a direct, line-by-line comparable codebase):

- `set_seed`, `get_device`, `random_train_validation_split`, `build_loader`, `mask_padding_item`, `assert_finite`
- `precision_mrr_at_k`, `evaluate`, `train_one_epoch`
- `run_dataset_experiment`

Per-epoch progress is appended to disk after every epoch, so partial runs on Kaggle are recoverable and progress is visible mid-run. Checkpoints are saved each time the validation `MRR@20` improves, plus a final checkpoint at the end of each dataset's training.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def random_train_validation_split(rows, validation_fraction=0.1, seed=42):
    indices = list(range(len(rows)))
    random.Random(seed).shuffle(indices)
    validation_size = max(1, int(len(indices) * validation_fraction))
    validation_indices = set(indices[:validation_size])

    train_rows = []
    validation_rows = []
    for index, row in enumerate(rows):
        if index in validation_indices:
            validation_rows.append(row)
        else:
            train_rows.append(row)
    return train_rows, validation_rows


def build_loader(rows, batch_size, shuffle, device):
    dataset = SessionGraphDataset(rows)
    use_cuda = device.type == "cuda"
    num_workers = 2 if use_cuda else 0
    return PyGDataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=use_cuda,
        persistent_workers=num_workers > 0,
    )


def mask_padding_item(logits):
    logits = logits.clone()
    logits[:, 0] = -torch.finfo(logits.dtype).max
    return logits


def assert_finite(name, tensor, batch_index):
    if not torch.isfinite(tensor).all():
        raise RuntimeError(f"Non-finite {name} detected in batch {batch_index}")

In [ ]:
def precision_mrr_at_k(logits, targets, k=20):
    k = min(k, logits.size(1))
    top_items = logits.topk(k, dim=1).indices
    matches = top_items.eq(targets.view(-1, 1))

    hits = matches.any(dim=1).float()
    ranks = torch.zeros(targets.size(0), device=logits.device)
    matched_rows, matched_cols = matches.nonzero(as_tuple=True)
    ranks[matched_rows] = matched_cols.float() + 1
    reciprocal_ranks = torch.where(ranks > 0, 1.0 / ranks, torch.zeros_like(ranks))

    return hits.sum().item(), reciprocal_ranks.sum().item(), targets.size(0)


@torch.no_grad()
def evaluate(model, loader, device, k=20):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    total_hits = 0.0
    total_mrr = 0.0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y)
        assert_finite("evaluation logits", logits, batch_index)
        assert_finite("evaluation loss", loss, batch_index)

        hits, mrr, examples = precision_mrr_at_k(logits, batch.y, k=k)
        total_loss += loss.item() * examples
        total_examples += examples
        total_hits += hits
        total_mrr += mrr

    return {
        "loss": total_loss / total_examples,
        "precision@20": 100.0 * total_hits / total_examples,
        "mrr@20": 100.0 * total_mrr / total_examples,
        "examples": total_examples,
    }


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch_index, batch in enumerate(loader):
        batch = batch.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        logits = mask_padding_item(model(batch))
        loss = F.cross_entropy(logits, batch.y)
        assert_finite("logits", logits, batch_index)
        assert_finite("loss", loss, batch_index)

        loss.backward()
        optimizer.step()

        examples = batch.y.size(0)
        total_loss += loss.item() * examples
        total_examples += examples

    return total_loss / total_examples

In [ ]:
DATASETS = {
    "Yoochoose 1/64": {
        "train_rows": YOOCHOOSE_1_64_TRAIN_ROWS,
        "test_rows": YOOCHOOSE_1_64_TEST_ROWS,
        "num_items": yoochoose_1_64_num_items,
    },
    "Diginetica": {
        "train_rows": DIGINETICA_TRAIN_ROWS,
        "test_rows": DIGINETICA_TEST_ROWS,
        "num_items": diginetica_num_items,
    },
}

config = {
    "epochs": 30,
    "patience": 5,
    "batch_size": 100,
    "learning_rate": 0.001,
    "weight_decay": 1e-5,
    "lr_decay_step": 3,
    "lr_decay_gamma": 0.1,
    "validation_fraction": 0.1,
    "hidden_dim": 100,
    "num_layers": 1,
    "num_heads": 4,
    "dropout": 0.1,
    "seed": 42,
}

MODEL_NAME = "GATSAGPool"
RESULT_FILE_PREFIX = "gat_sagpool"

device = get_device()
print(f"device={device}")
print(config)

In [ ]:
def checkpoint_name(dataset_name):
    safe_name = dataset_name.lower().replace(" ", "_").replace("/", "_")
    return f"{RESULT_FILE_PREFIX}_{safe_name}.pt"


def _save_history_so_far(history_rows, history_path):
    pd.DataFrame(history_rows).to_csv(history_path, index=False)


def run_dataset_experiment(
    dataset_name,
    dataset,
    config,
    device,
    model_factory,
    model_name=MODEL_NAME,
    result_file_prefix=RESULT_FILE_PREFIX,
):
    """Train a session-graph model on a single dataset.

    `model_factory(num_items, config)` must return an `nn.Module` that
    accepts a PyG batch (with the fields produced by `build_session_graph`)
    and returns logits of shape `[batch_size, num_items]`. The runner is
    intentionally model-agnostic so SR-GNN, TAGNN, and SAGPool variants
    share the same training harness.
    """
    set_seed(config["seed"])
    train_rows = list(dataset["train_rows"])
    test_rows = list(dataset["test_rows"])
    train_rows, validation_rows = random_train_validation_split(
        train_rows,
        validation_fraction=config["validation_fraction"],
        seed=config["seed"],
    )

    train_loader = build_loader(
        train_rows, config["batch_size"], shuffle=True, device=device
    )
    validation_loader = build_loader(
        validation_rows, config["batch_size"], shuffle=False, device=device
    )
    test_loader = build_loader(
        test_rows, config["batch_size"], shuffle=False, device=device
    )

    model = model_factory(num_items=dataset["num_items"], config=config).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=config["lr_decay_step"],
        gamma=config["lr_decay_gamma"],
    )

    history = []
    best_validation_mrr = -1.0
    best_validation_precision = -1.0
    best_epoch = 0
    best_state = None
    bad_counter = 0

    checkpoint_path = CHECKPOINTS_DIR / checkpoint_name(dataset_name)
    history_path = RESULTS_DIR / f"{result_file_prefix}_history.csv"

    for epoch in range(1, config["epochs"] + 1):
        started_at = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        validation_metrics = evaluate(model, validation_loader, device)
        scheduler.step()

        row = {
            "dataset": dataset_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_metrics["loss"],
            "validation_precision@20": validation_metrics["precision@20"],
            "validation_mrr@20": validation_metrics["mrr@20"],
            "epoch_seconds": time.time() - started_at,
            "train_examples": len(train_rows),
            "validation_examples": len(validation_rows),
            "test_examples": len(test_rows),
        }
        history.append(row)
        print(
            f"{dataset_name} epoch {epoch:02d} "
            f"loss={train_loss:.4f} "
            f"val_loss={row['validation_loss']:.4f} "
            f"val_P@20={row['validation_precision@20']:.2f} "
            f"val_MRR@20={row['validation_mrr@20']:.2f} "
            f"time={row['epoch_seconds']:.1f}s"
        )

        _save_history_so_far(history, history_path)

        if validation_metrics["mrr@20"] >= best_validation_mrr:
            best_validation_mrr = validation_metrics["mrr@20"]
            best_validation_precision = validation_metrics["precision@20"]
            best_epoch = epoch
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            bad_counter = 0
            torch.save(
                {
                    "dataset": dataset_name,
                    "model": model_name,
                    "config": dict(config),
                    "num_items": dataset["num_items"],
                    "state_dict": best_state,
                    "best_epoch": best_epoch,
                    "best_validation_precision@20": best_validation_precision,
                    "best_validation_mrr@20": best_validation_mrr,
                },
                checkpoint_path,
            )
        else:
            bad_counter += 1
            if bad_counter >= config["patience"]:
                print(f"early stopping at epoch {epoch}; best epoch was {best_epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, device)

    torch.save(
        {
            "dataset": dataset_name,
            "model": model_name,
            "config": dict(config),
            "num_items": dataset["num_items"],
            "state_dict": model.state_dict(),
            "test_metrics": test_metrics,
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_precision@20": best_validation_precision,
            "best_validation_mrr@20": best_validation_mrr,
        },
        checkpoint_path,
    )

    result = {
        "dataset": dataset_name,
        "test_precision@20": test_metrics["precision@20"],
        "test_mrr@20": test_metrics["mrr@20"],
        "test_loss": test_metrics["loss"],
        "best_epoch": best_epoch,
        "best_validation_precision@20": best_validation_precision,
        "best_validation_mrr@20": best_validation_mrr,
        "train_examples": len(train_rows),
        "validation_examples": len(validation_rows),
        "test_examples": len(test_rows),
        "num_items": dataset["num_items"],
        "checkpoint_path": str(checkpoint_path),
    }
    return result, history

## Bidirectional GAT Encoder

The encoder is a direct port of the bidirectional GAT stack used by `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`. Keeping the encoder identical across the three branches isolates the architectural difference of this notebook to the readout (SAGPool), so any change in `Precision@20` / `MRR@20` is attributable to the readout choice rather than to encoder differences.

Defaults match the other two notebooks:

- hidden size `100`
- one GAT layer per direction
- `4` attention heads, per-head output `25`, concatenated back to `100`
- dropout `0.1` inside the GAT layer and after activation
- ELU activation
- residual connection inside each directional stack
- self-loops added by `GATConv`
- normalized transition counts passed as one-dimensional edge attributes
- forward / backward states concatenated, then fused by a linear `200 -> 100`

In [ ]:
class DirectionalGATStack(nn.Module):
    """GAT stack for one transition direction."""

    def __init__(
        self,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        concat_heads=True,
        residual=True,
    ):
        super().__init__()

        if concat_heads and hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim ({hidden_dim}) must be divisible by num_heads "
                f"({num_heads}) when concat_heads=True"
            )

        self.dropout = dropout
        self.residual = residual
        per_head_dim = hidden_dim // num_heads if concat_heads else hidden_dim
        self.gat_layers = nn.ModuleList(
            GATConv(
                in_channels=hidden_dim,
                out_channels=per_head_dim,
                heads=num_heads,
                concat=concat_heads,
                dropout=dropout,
                add_self_loops=True,
                edge_dim=1,
            )
            for _ in range(num_layers)
        )

    def forward(self, node_features, edge_index, edge_weight=None):
        edge_attr = edge_weight.unsqueeze(-1) if edge_weight is not None else None
        for gat_layer in self.gat_layers:
            previous = node_features
            node_features = F.elu(gat_layer(node_features, edge_index, edge_attr))
            node_features = F.dropout(
                node_features, p=self.dropout, training=self.training
            )
            if self.residual:
                node_features = node_features + previous
        return node_features


class BidirectionalGATEncoder(nn.Module):
    """Separate forward/backward GAT stacks with shared item embeddings."""

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)
        self.forward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.backward_gat = DirectionalGATStack(
            hidden_dim, num_layers, num_heads, dropout
        )
        self.direction_fusion = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)

    def forward(
        self,
        node_item_ids,
        forward_edge_index,
        forward_edge_weight,
        backward_edge_index,
        backward_edge_weight,
    ):
        node_features = self.embedding(node_item_ids)
        forward_features = self.forward_gat(
            node_features, forward_edge_index, forward_edge_weight
        )
        backward_features = self.backward_gat(
            node_features, backward_edge_index, backward_edge_weight
        )
        return self.direction_fusion(
            torch.cat([forward_features, backward_features], dim=-1)
        )

## GAT-SAGPool Model

Encoder is `BidirectionalGATEncoder` from above; readout is a single-block SAGPool head with last-click signal preservation.

### Readout (`B2`)

1. Score every node with a `GraphConv` scoring layer (the SAGPool default), producing one scalar per node.
2. Retain the top fraction of nodes per graph (per-graph top-k), using `sagpool_ratio` as the hyperparameter (default `0.5`). PyG's `SAGPooling` guarantees that every graph keeps at least one node, including degenerate cases like single-item prefixes.
3. Aggregate the retained nodes per graph by concatenating `global_mean_pool` and `global_max_pool` outputs (`2 * hidden_dim`).
4. Concatenate the aggregated SAGPool readout with the **last-click contextual node embedding** taken from the un-pooled encoder output. SAGPool is permutation invariant by construction and does not see ordering information, so explicitly carrying the last-click vector is what protects the immediate intent signal of the session.
5. Project the combined `3 * hidden_dim` vector down to `hidden_dim`. This keeps the final session embedding compatible with the shared item embedding table.

### Scoring head (`B3`)

The session embedding is multiplied by the shared item embedding matrix to produce logits of shape `[batch_size, num_items]`. The padding item id `0` is excluded from ranking by `mask_padding_item` (defined earlier and shared across the three notebooks), so the candidate space is exactly the remapped train item ids `1 .. num_items - 1`. The padding row of the embedding table is also explicitly zeroed in `reset_parameters` and held at zero by `padding_idx=0` on `nn.Embedding`.

In [ ]:
class GATSAGPool(nn.Module):
    """Bidirectional GAT encoder + SAGPool readout for session recommendation.

    Architecture summary:

      session prefix
          |
      directed session graph (forward + backward edges, normalized weights)
          |
      BidirectionalGATEncoder  ->  contextual node embeddings  [N, H]
          |                                          \\
          |                                           +--> last_click_repr [B, H]
          v
      SAGPooling (GraphConv scorer, top-`ratio`)  ->  pooled nodes
          |
      [global_mean_pool || global_max_pool]  ->  pooled session repr [B, 2H]
          |
      concat(pooled_session, last_click_repr)  ->  [B, 3H]
          |
      Linear(3H -> H)  ->  session_repr [B, H]
          |
      session_repr @ item_embedding.T  ->  logits [B, num_items]
    """

    def __init__(
        self,
        num_items,
        hidden_dim=100,
        num_layers=1,
        num_heads=4,
        dropout=0.1,
        sagpool_ratio=0.5,
    ):
        super().__init__()
        self.encoder = BidirectionalGATEncoder(
            num_items, hidden_dim, num_layers, num_heads, dropout
        )
        self.hidden_dim = hidden_dim
        self.sagpool_ratio = sagpool_ratio

        self.sagpool = SAGPooling(
            in_channels=hidden_dim,
            ratio=sagpool_ratio,
            GNN=GraphConv,
        )

        self.session_projection = nn.Linear(3 * hidden_dim, hidden_dim, bias=True)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_dim)
        for parameter in self.parameters():
            parameter.data.uniform_(-stdv, stdv)
        with torch.no_grad():
            self.encoder.embedding.weight[0].fill_(0)

    def forward(self, batch):
        """Return logits `[batch_size, num_items]`."""
        node_hidden = self.encoder(
            batch.x,
            batch.forward_edge_index,
            batch.forward_edge_weight,
            batch.backward_edge_index,
            batch.backward_edge_weight,
        )

        pooled_x, _, _, pooled_batch, _, _ = self.sagpool(
            node_hidden,
            batch.forward_edge_index,
            None,
            batch.batch,
        )

        mean_session = global_mean_pool(
            pooled_x, pooled_batch, size=batch.num_graphs
        )
        max_session = global_max_pool(
            pooled_x, pooled_batch, size=batch.num_graphs
        )

        graph_node_offsets = batch.ptr[:-1]
        last_click_local = batch.last_click.view(-1).long()
        last_click_abs = graph_node_offsets + last_click_local
        last_click_repr = node_hidden[last_click_abs]

        session_repr = self.session_projection(
            torch.cat([mean_session, max_session, last_click_repr], dim=-1)
        )

        item_embeddings = self.encoder.embedding.weight
        logits = session_repr @ item_embeddings.T
        return logits

In [ ]:
def gat_sagpool_factory(num_items, config):
    """Build a `GATSAGPool` model from the shared experiment config dict.

    Used by `run_dataset_experiment(..., model_factory=gat_sagpool_factory)`.
    """
    return GATSAGPool(
        num_items=num_items,
        hidden_dim=config["hidden_dim"],
        num_layers=config["num_layers"],
        num_heads=config["num_heads"],
        dropout=config["dropout"],
        sagpool_ratio=config.get("sagpool_ratio", 0.5),
    )


config.setdefault("sagpool_ratio", 0.5)

## Numerical stability and shape validation

Before launching a full training run on Kaggle (which is expensive), this section runs an inexpensive smoke test on a handcrafted mini-batch covering the edge cases of the session-graph dataset:

- a normal length-5 prefix
- a prefix with a repeated item
- a length-1 prefix (minimum graph: 1 node, 0 edges)
- a prefix containing a cycle / repeated transitions
- a longer length-8 prefix

For this batch the smoke test asserts:

- the per-graph contract from `build_session_graph` (1D `x`, two-row `forward_edge_index` / `backward_edge_index`, scalar `last_click` matching `sequence[-1]`)
- the per-batch contract from PyG batching (`batch.num_graphs`, `batch.y`, `batch.last_click`, `batch.sequence_length` are all `[batch_size]`)
- the model forward pass returns logits of shape `[batch_size, num_items]`
- all logits are finite
- after `mask_padding_item`, the padding id `0` is never selected by `argmax`
- the backward pass works for variable-size graphs and produces a finite loss

If any assertion trips, training is aborted immediately rather than burning Kaggle GPU time on a broken model.

In [ ]:
def _run_gat_sagpool_smoke_test():
    set_seed(0)

    smoke_num_items = 50
    sample_rows = [
        ([1, 2, 3, 4, 5], 6),
        ([1, 1, 2], 3),
        ([7], 8),
        ([1, 2, 3, 1, 2], 4),
        ([5, 6, 7, 8, 9, 10, 11, 12], 13),
    ]
    batch_size = len(sample_rows)

    single = build_session_graph(*sample_rows[0])
    assert single.x.dim() == 1, f"x must be 1D, got {tuple(single.x.shape)}"
    assert single.forward_edge_index.size(0) == 2, "forward_edge_index must have 2 rows"
    assert single.backward_edge_index.size(0) == 2, "backward_edge_index must have 2 rows"
    assert single.num_nodes == single.x.numel(), "num_nodes must equal len(x)"
    assert int(single.last_click.item()) == int(single.sequence[-1].item()), (
        "last_click must match the last position of the sequence"
    )

    dataset = SessionGraphDataset(sample_rows)
    loader = PyGDataLoader(dataset, batch_size=batch_size, shuffle=False)
    batch = next(iter(loader)).to(device)

    assert batch.num_graphs == batch_size, batch.num_graphs
    assert tuple(batch.y.shape) == (batch_size,), batch.y.shape
    assert tuple(batch.last_click.shape) == (batch_size,), batch.last_click.shape
    assert tuple(batch.sequence_length.shape) == (batch_size,), batch.sequence_length.shape

    model = gat_sagpool_factory(num_items=smoke_num_items, config=config).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(batch)
        masked = mask_padding_item(logits)

    assert tuple(logits.shape) == (batch_size, smoke_num_items), (
        f"expected {(batch_size, smoke_num_items)}, got {tuple(logits.shape)}"
    )
    assert torch.isfinite(logits).all(), "non-finite logits in smoke test"
    assert torch.isfinite(masked).all(), "non-finite masked logits in smoke test"

    top_per_row = masked.argmax(dim=1)
    assert (top_per_row > 0).all(), "argmax must never pick padding id 0 after masking"

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    optimizer.zero_grad(set_to_none=True)
    train_logits = mask_padding_item(model(batch))
    loss = F.cross_entropy(train_logits, batch.y)
    assert torch.isfinite(loss), f"non-finite training loss: {loss}"
    loss.backward()
    optimizer.step()

    print(
        f"smoke_test_passed batch={batch_size} num_items={smoke_num_items} "
        f"logits_shape={tuple(logits.shape)} backward_loss={loss.item():.4f}"
    )


_run_gat_sagpool_smoke_test()

## Training Run 

This section actually trains `GATSAGPool` on `Yoochoose 1/64` and `Diginetica` using the model-agnostic `run_dataset_experiment`.

- **C1 - config parity**: Adam with `lr=0.001`, `weight_decay=1e-5`, `StepLR(step_size=3, gamma=0.1)`, batch size `100`, early-stopping patience `5` (matches `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`).
- **C2 - validation protocol**: deterministic `10%` random split from the training rows via `random_train_validation_split`. Each epoch records `train_loss`, `validation_loss`, `validation_precision@20`, and `validation_mrr@20` into the per-dataset history CSV.
- **C3 - checkpointing**: best epoch is selected by `validation_mrr@20` (consistent with the SR-GNN / TAGNN notebooks). The best state is persisted to `results/checkpoints/gat_sagpool_<dataset>.pt` as soon as it is observed, and overwritten at the end of training with the best weights plus the final test metrics. The path is also surfaced through the per-dataset result row, so the aggregated results CSV doubles as a "where to find the best model" registry.

Per-epoch history is streamed to `results/gat_sagpool_history.csv` after every epoch, so a partial Kaggle run is still useful.

In [ ]:
all_results = []
all_history = []

for dataset_name, dataset in DATASETS.items():
    print(f"=== training {dataset_name} ===")
    result, history = run_dataset_experiment(
        dataset_name, dataset, config, device, gat_sagpool_factory
    )
    all_results.append(result)
    all_history.extend(history)
    print(
        f"=== finished {dataset_name}: "
        f"test_P@20={result['test_precision@20']:.2f} "
        f"test_MRR@20={result['test_mrr@20']:.2f} "
        f"best_epoch={result['best_epoch']} "
        f"checkpoint={result['checkpoint_path']} ==="
    )

results = pd.DataFrame(all_results)
history = pd.DataFrame(all_history)

results_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_results.csv"
history_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_history.csv"
results.to_csv(results_path, index=False)
history.to_csv(history_path, index=False)

try:
    display(results)
except NameError:
    print(results.to_string(index=False))
print(f"saved {results_path}")
print(f"saved {history_path}")

## Checkpoint Summary

For each dataset this writes a compact summary that combines:

- `best_epoch` (the epoch that won the validation `MRR@20` competition),
- the best validation `Precision@20` and `MRR@20` at that epoch,
- the final test `Precision@20`, `MRR@20`, and `loss` (evaluated after restoring the best weights),
- the path to the best checkpoint on disk.

The same physical file holds both the best-by-validation weights and the final test-time metrics (the runner restores best weights before computing test metrics, then overwrites the checkpoint with the best state and those test metrics attached). Saving an explicit summary CSV makes the "where is the best model" registry explicit and easy to consume from the cross-notebook comparison tables under `results/`.

In [ ]:
checkpoint_summary_rows = []
for record in all_results:
    checkpoint_summary_rows.append(
        {
            "dataset": record["dataset"],
            "best_epoch": record["best_epoch"],
            "best_validation_precision@20": record["best_validation_precision@20"],
            "best_validation_mrr@20": record["best_validation_mrr@20"],
            "test_precision@20": record["test_precision@20"],
            "test_mrr@20": record["test_mrr@20"],
            "test_loss": record["test_loss"],
            "best_checkpoint_path": record["checkpoint_path"],
            "final_checkpoint_path": record["checkpoint_path"],
        }
    )

checkpoint_summary = pd.DataFrame(checkpoint_summary_rows)
checkpoint_summary_path = (
    RESULTS_DIR / f"{RESULT_FILE_PREFIX}_checkpoint_summary.csv"
)
checkpoint_summary.to_csv(checkpoint_summary_path, index=False)

try:
    display(checkpoint_summary)
except NameError:
    print(checkpoint_summary.to_string(index=False))
print(f"saved {checkpoint_summary_path}")

## Reporting


This is the reporting layer for `GATSAGPool`.



The cross-model writes are intentionally non-invasive: they never delete other models' rows and never touch a file unless its schema is recognized.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results.plot.bar(
    x="dataset",
    y="test_precision@20",
    ax=axes[0],
    legend=False,
    color="#3b6ea8",
    title="Test Precision@20 (GAT-SAGPool)",
)
axes[0].set_ylabel("%")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

results.plot.bar(
    x="dataset",
    y="test_mrr@20",
    ax=axes[1],
    legend=False,
    color="#b45f3c",
    title="Test MRR@20 (GAT-SAGPool)",
)
axes[1].set_ylabel("%")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
figure_path = RESULTS_DIR / f"{RESULT_FILE_PREFIX}_metrics.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
print(f"saved {figure_path}")

## Cross-model integration

Two outputs are produced here:

1. `results/gat_sagpool_model_summary_rows.csv` - a self-contained CSV with the same schema as `results/model_summary.csv` and a `model="GAT-SAGPool"` column added. This is always written and is the file you would paste into the central comparison table during an offline merge.
2. If `results/model_summary.csv` already exists in `RESULTS_DIR` (e.g. uploaded as Kaggle input alongside this run), it is rewritten in place: any prior `GAT-SAGPool` rows are dropped, and the fresh ones are appended. Other models' rows are never touched. If no prior file exists, the SAGPool rows bootstrap a new `model_summary.csv`.

In addition, when both `results/gat_sr_gnn_results.csv` and `results/gat_tagnn_results.csv` happen to be present in `RESULTS_DIR`, a 3-way head-to-head comparison is written to `results/gat_three_way_comparison.csv` so SAGPool can be inspected next to the other two GAT branches without disturbing the existing 2-way `gat_head_to_head.csv`.

In [ ]:
SAGPOOL_MODEL_LABEL = "GAT-SAGPool"

model_summary_additions = pd.DataFrame(
    [{"model": SAGPOOL_MODEL_LABEL, **row} for row in all_results]
)
model_summary_additions_path = (
    RESULTS_DIR / f"{RESULT_FILE_PREFIX}_model_summary_rows.csv"
)
model_summary_additions.to_csv(model_summary_additions_path, index=False)
print(f"saved {model_summary_additions_path}")

existing_summary_path = RESULTS_DIR / "model_summary.csv"
if existing_summary_path.exists():
    existing_summary = pd.read_csv(existing_summary_path)
    if "model" not in existing_summary.columns:
        print(
            f"existing summary at {existing_summary_path} has no 'model' column; "
            "skipping merge to avoid corrupting the file"
        )
    else:
        kept = existing_summary[existing_summary["model"] != SAGPOOL_MODEL_LABEL]
        merged = pd.concat([kept, model_summary_additions], ignore_index=True)
        merged.to_csv(existing_summary_path, index=False)
        print(f"updated {existing_summary_path}")
else:
    model_summary_additions.to_csv(existing_summary_path, index=False)
    print(f"created {existing_summary_path}")

sr_gnn_results_path = RESULTS_DIR / "gat_sr_gnn_results.csv"
tagnn_results_path = RESULTS_DIR / "gat_tagnn_results.csv"
if sr_gnn_results_path.exists() and tagnn_results_path.exists():
    sr_gnn_df = pd.read_csv(sr_gnn_results_path).set_index("dataset")
    tagnn_df = pd.read_csv(tagnn_results_path).set_index("dataset")
    sagpool_df = pd.DataFrame(all_results).set_index("dataset")

    head_to_head_rows = []
    for dataset_name in sagpool_df.index:
        if dataset_name not in sr_gnn_df.index or dataset_name not in tagnn_df.index:
            continue
        head_to_head_rows.append(
            {
                "dataset": dataset_name,
                "gat_sr_gnn_precision@20": sr_gnn_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_tagnn_precision@20": tagnn_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_sagpool_precision@20": sagpool_df.loc[
                    dataset_name, "test_precision@20"
                ],
                "gat_sr_gnn_mrr@20": sr_gnn_df.loc[dataset_name, "test_mrr@20"],
                "gat_tagnn_mrr@20": tagnn_df.loc[dataset_name, "test_mrr@20"],
                "gat_sagpool_mrr@20": sagpool_df.loc[dataset_name, "test_mrr@20"],
            }
        )

    if head_to_head_rows:
        head_to_head = pd.DataFrame(head_to_head_rows)
        head_to_head_path = RESULTS_DIR / "gat_three_way_comparison.csv"
        head_to_head.to_csv(head_to_head_path, index=False)
        try:
            display(head_to_head)
        except NameError:
            print(head_to_head.to_string(index=False))
        print(f"saved {head_to_head_path}")
    else:
        print("no overlapping datasets for 3-way comparison; skipping")
else:
    missing = [
        str(path)
        for path in (sr_gnn_results_path, tagnn_results_path)
        if not path.exists()
    ]
    print(f"3-way comparison skipped; missing baselines: {missing}")

## Best-checkpoint upload (optional)

This mirrors the final cell of `gat_sr_gnn_architecture.ipynb` and `gat_tagnn_architecture.ipynb`: the best checkpoint (chosen by validation `MRR@20`) is published to KaggleHub under a SAGPool-specific slug. The call is wrapped so that an authentication failure or running outside Kaggle does not break the notebook flow - in that case the checkpoint stays available locally under `RESULTS_DIR / "checkpoints"`.

In [ ]:
best_result = results.sort_values(
    ["best_validation_mrr@20", "best_validation_precision@20"],
    ascending=False,
).iloc[0]
best_dataset_slug = (
    best_result["dataset"].lower().replace(" ", "-").replace("/", "-")
)
best_checkpoint_path = Path(best_result["checkpoint_path"])

MODEL_SLUG = "gat-sagpool-session-recommender"
VARIATION_SLUG = f"best-{best_dataset_slug}"

try:
    kagglehub.model_upload(
        handle=f"karolbystrek/{MODEL_SLUG}/pytorch/{VARIATION_SLUG}",
        local_model_dir=str(best_checkpoint_path.parent),
        version_notes=(
            "Best GAT-SAGPool checkpoint selected by validation MRR@20. "
            f"Update {date.today().isoformat()}"
        ),
    )
    print(f"uploaded {MODEL_SLUG}/{VARIATION_SLUG} from {best_checkpoint_path.parent}")
except Exception as exc:
    print(f"kagglehub upload skipped ({exc.__class__.__name__}): {exc}")